# Notebook 06 — Qualidade de Dados e Análise

**MVP de Engenharia de Dados** · PUC-Rio · Sprint 3

---

### Objetivo

Verificar a consistência física dos dados modelados e responder às cinco perguntas de negócio definidas no início do trabalho.

### Entrada e saída

| | |
|---|---|
| **Entrada** | Esquema estrela em `workspace.gold` |
| **Saída** | `workspace.gold.qa_violacoes_balanco` e as respostas às perguntas |

### Estrutura

**Parte 1 — Qualidade de dados.** Formula e testa uma identidade física do sistema elétrico, investiga as exceções encontradas e converte a verificação em regra permanente do pipeline.

**Parte 2 — Análise.** Responde às cinco perguntas usando SQL sobre o esquema estrela.

# PARTE 1 — Qualidade de Dados

## 1. A identidade do balanço energético

Para cada subsistema, em cada hora, a energia gerada deve igualar a energia consumida mais a energia exportada: **geração = carga + intercâmbio**.

A hipótese surgiu da observação de um registro isolado — Norte, 01/01/2019 às 00:00: geração 8.856,029 menos carga 4.888,033 resulta em 3.967,996, valor idêntico ao intercâmbio registrado.

Duas amostras não provam uma regra. Esta célula testa a identidade sobre os 245.472 registros da base.

Além de validar o pipeline, o teste resolve uma ambiguidade que o dicionário do ONS não explicita: o **sinal do intercâmbio**. Confirmada a identidade, valores positivos significam exportação e negativos, importação. Sem essa determinação, a resposta da Pergunta 5 sairia invertida.

In [0]:
%sql
-- Hipótese a testar: carga = geração - intercâmbio.
-- A amostra do Norte em 2019-01-01 00:00 sugere a identidade (geração 8.856,029
-- menos carga 4.888,033 = 3.967,996, que é o valor de val_intercambio), mas duas
-- amostras não provam uma regra. Esta célula verifica a hipótese em toda a base e
-- mede o erro máximo, em vez de assumi-la.
WITH g AS (
  SELECT sk_tempo, sk_subsistema, SUM(val_geracao_mwmed) AS geracao
  FROM workspace.gold.fato_geracao GROUP BY 1, 2
)
SELECT
  COUNT(*)                                                         AS registros_conferidos,
  ROUND(MAX(ABS(g.geracao - c.val_carga_mwmed - c.val_intercambio_mwmed)), 6) AS erro_maximo,
  SUM(CASE WHEN ABS(g.geracao - c.val_carga_mwmed - c.val_intercambio_mwmed) > 1
           THEN 1 ELSE 0 END)                                      AS violacoes
FROM g
JOIN workspace.gold.fato_carga c
  ON c.sk_tempo = g.sk_tempo AND c.sk_subsistema = g.sk_subsistema;

registros_conferidos,erro_maximo,violacoes
245472,1551.319997,24


## 2. Investigação das violações

A identidade se confirma em 245.448 registros e **falha em 24** — 0,0098% da base, com resíduo máximo de 1.551 MWmed.

As três consultas a seguir reduzem o espaço de suspeitos:

| Consulta | Pergunta que responde |
|---|---|
| Listagem das violações | Onde elas estão? |
| Geração por fonte nos dias vizinhos | Alguma fonte sumiu do registro? |
| Janela de nove dias | Foi a carga ou o intercâmbio que se deslocou? |

In [0]:
%sql
WITH g AS (
  SELECT sk_tempo, sk_subsistema, SUM(val_geracao_mwmed) AS geracao
  FROM workspace.gold.fato_geracao GROUP BY 1, 2
)
SELECT t.data, t.hora, s.nom_subsistema,
  ROUND(g.geracao, 2)                AS geracao,
  ROUND(c.val_carga_mwmed, 2)        AS carga,
  ROUND(c.val_intercambio_mwmed, 2)  AS intercambio,
  ROUND(g.geracao - c.val_carga_mwmed - c.val_intercambio_mwmed, 2) AS residuo
FROM g
JOIN workspace.gold.fato_carga    c ON c.sk_tempo = g.sk_tempo AND c.sk_subsistema = g.sk_subsistema
JOIN workspace.gold.dim_tempo      t ON t.sk_tempo      = g.sk_tempo
JOIN workspace.gold.dim_subsistema s ON s.sk_subsistema = g.sk_subsistema
WHERE ABS(g.geracao - c.val_carga_mwmed - c.val_intercambio_mwmed) > 1
ORDER BY t.data, t.hora, s.nom_subsistema;

data,hora,nom_subsistema,geracao,carga,intercambio,residuo
2022-09-14,0,SUL,15210.82,10078.4,6518.23,-1385.82
2022-09-14,1,SUL,14555.75,9410.23,6491.03,-1345.51
2022-09-14,2,SUL,14137.73,9068.26,6413.48,-1344.02
2022-09-14,3,SUL,13825.55,8974.42,6195.16,-1344.03
2022-09-14,4,SUL,14322.29,9171.59,6494.63,-1343.92
2022-09-14,5,SUL,14515.66,9788.18,6071.47,-1343.99
2022-09-14,6,SUL,15089.65,11059.63,5388.6,-1358.58
2022-09-14,7,SUL,15676.81,12133.27,4903.72,-1360.18
2022-09-14,8,SUL,16676.22,12882.45,5153.79,-1360.01
2022-09-14,9,SUL,16715.51,12904.3,5258.62,-1447.41


**Resultado:** todas as 24 violações pertencem ao subsistema **Sul**, no dia **14/09/2022**, cobrindo as 24 horas do dia. O resíduo fica entre −1.069 e −1.551 MWmed enquanto a carga varia de 8.974 a 14.836.

Isso elimina duas hipóteses de imediato.

**Não é erro do pipeline.** Um join duplicando ou um filtro errado espalharia violações por toda a base, não as concentraria em 24 horas consecutivas de um único subsistema.

**Não é ruído de medição.** Ruído seria aleatório em sinal e magnitude, não um déficit quase constante ao longo de um dia inteiro.

Resta a hipótese de que alguma fonte de geração tenha desaparecido do registro naquele dia. A consulta seguinte compara cada fonte com os dias vizinhos.

In [0]:
%sql
-- Compara a geração do Sul por fonte nos dias ao redor de 14/09/2022.
-- Se uma fonte específica despencar só no dia 14, ela é a origem do resíduo.
SELECT t.data, f.nom_fonte, ROUND(AVG(g.val_geracao_mwmed), 1) AS media_mwmed
FROM workspace.gold.fato_geracao g
JOIN workspace.gold.dim_tempo      t ON t.sk_tempo      = g.sk_tempo
JOIN workspace.gold.dim_subsistema s ON s.sk_subsistema = g.sk_subsistema
JOIN workspace.gold.dim_fonte      f ON f.sk_fonte      = g.sk_fonte
WHERE s.id_subsistema = 'S'
  AND t.data BETWEEN '2022-09-12' AND '2022-09-16'
GROUP BY t.data, f.nom_fonte
ORDER BY f.nom_fonte, t.data;

data,nom_fonte,media_mwmed
2022-09-12,Eólica,768.2
2022-09-13,Eólica,851.7
2022-09-14,Eólica,833.1
2022-09-15,Eólica,440.0
2022-09-16,Eólica,1128.9
2022-09-12,Fotovoltaica,1.7
2022-09-13,Fotovoltaica,1.7
2022-09-14,Fotovoltaica,1.6
2022-09-15,Fotovoltaica,1.6
2022-09-16,Fotovoltaica,1.3


**Resultado:** nenhuma fonte sumiu.

| Fonte | 13/09 | **14/09** | 15/09 |
|---|---|---|---|
| Eólica | 851,7 | **833,1** | 440,0 |
| Fotovoltaica | 1,7 | **1,6** | 1,6 |
| Hidráulica | 12.007,5 | **13.613,0** | 12.965,6 |
| Térmica | 1.417,0 | **1.371,4** | 1.367,1 |

Todas dentro da faixa dos dias vizinhos — e a hidráulica no dia 14 é a **maior** dos cinco dias analisados. Somadas, resultam em cerca de 15.819 MWmed, compatível com a geração registrada.

Não há evidência de que alguma fonte tenha desaparecido do registro. O desvio mais provável, portanto, está do outro lado da equação: em carga ou em intercambio. A consulta a seguir abre a janela para nove dias e mostra os três valores lado a lado.

In [0]:
%sql
-- Geração, carga e intercâmbio do Sul em torno de 14/09/2022.
-- Como a geração já foi descartada como causa, o desvio deve estar
-- em carga ou em intercâmbio. Esta consulta mostra qual dos dois se desloca.
WITH g AS (
  SELECT sk_tempo, sk_subsistema, SUM(val_geracao_mwmed) AS geracao
  FROM workspace.gold.fato_geracao GROUP BY 1, 2
)
SELECT t.data,
  ROUND(AVG(g.geracao), 1)                AS geracao_media,
  ROUND(AVG(c.val_carga_mwmed), 1)        AS carga_media,
  ROUND(AVG(c.val_intercambio_mwmed), 1)  AS intercambio_medio,
  ROUND(AVG(g.geracao - c.val_carga_mwmed - c.val_intercambio_mwmed), 1) AS residuo_medio
FROM g
JOIN workspace.gold.fato_carga     c ON c.sk_tempo = g.sk_tempo AND c.sk_subsistema = g.sk_subsistema
JOIN workspace.gold.dim_tempo      t ON t.sk_tempo      = g.sk_tempo
JOIN workspace.gold.dim_subsistema s ON s.sk_subsistema = g.sk_subsistema
WHERE s.id_subsistema = 'S'
  AND t.data BETWEEN '2022-09-10' AND '2022-09-18'
GROUP BY t.data
ORDER BY t.data;

data,geracao_media,carga_media,intercambio_medio,residuo_medio
2022-09-10,15043.3,10175.8,4867.5,0.0
2022-09-11,12614.0,8938.5,3675.5,0.0
2022-09-12,14469.2,11897.0,2572.2,0.0
2022-09-13,14278.0,11915.6,2362.4,0.0
2022-09-14,15819.2,12033.1,5103.6,-1317.6
2022-09-15,14774.3,11837.3,2937.0,0.0
2022-09-16,14324.2,11473.5,2850.7,0.0
2022-09-17,12929.9,9486.8,3443.1,0.0
2022-09-18,10446.3,8077.7,2368.6,0.0


## 3. Conclusão e decisão

O resíduo é **exatamente zero em oito dos nove dias** e −1.317,6 apenas no dia 14.

| | 12/09 | 13/09 | **14/09** | 15/09 | 16/09 |
|---|---|---|---|---|---|
| Carga | 11.897 | 11.916 | **12.033** | 11.837 | 11.474 |
| Intercâmbio | 2.572 | 2.362 | **5.104** | 2.937 | 2.851 |

A carga permanece normal. O **intercâmbio praticamente dobra** — é o campo que se desloca.

Ainda assim, a causa raiz **não é determinável** com os dados disponíveis: tanto um intercâmbio superestimado quanto uma geração subnotificada em 1.317 MWmed explicariam o resíduo. O registro se limita ao que a evidência sustenta.

**Impacto:** 24 registros em 245.472, num único dia de 2.557. A discrepância equivale a 31,6 GWh, contra 607,5 TWh gerados no país em 2022 — cerca de 0,005% do total anual. Nenhuma conclusão da análise é afetada.

**Decisão: manter os registros sem alteração.** Corrigir dado oficial sem conhecer a causa seria inventar valores; removê-los quebraria a completude comprovada da série. A anomalia é registrada numa tabela de auditoria que preserva os valores originais e o resíduo medido.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.qa_violacoes_balanco
COMMENT 'Tabela de auditoria de qualidade. Registra os instantes em que a identidade
geracao = carga + intercambio nao se fecha, com o residuo medido. Nenhum dado de origem
e alterado ou removido: as violacoes sao apenas registradas aqui para rastreabilidade.'
AS
WITH g AS (
  SELECT sk_tempo, sk_subsistema, SUM(val_geracao_mwmed) AS geracao
  FROM workspace.gold.fato_geracao GROUP BY 1, 2
)
SELECT t.data, t.hora, s.id_subsistema, s.nom_subsistema,
  ROUND(g.geracao, 4)               AS geracao_mwmed,
  ROUND(c.val_carga_mwmed, 4)       AS carga_mwmed,
  ROUND(c.val_intercambio_mwmed, 4) AS intercambio_mwmed,
  ROUND(g.geracao - c.val_carga_mwmed - c.val_intercambio_mwmed, 4) AS residuo_mwmed,
  current_timestamp()               AS data_verificacao
FROM g
JOIN workspace.gold.fato_carga     c ON c.sk_tempo = g.sk_tempo AND c.sk_subsistema = g.sk_subsistema
JOIN workspace.gold.dim_tempo      t ON t.sk_tempo      = g.sk_tempo
JOIN workspace.gold.dim_subsistema s ON s.sk_subsistema = g.sk_subsistema
WHERE ABS(g.geracao - c.val_carga_mwmed - c.val_intercambio_mwmed) > 1;

num_affected_rows,num_inserted_rows


### Documentação das colunas da tabela de auditoria

A tabela é recriada com `CREATE OR REPLACE` a cada execução, o que apaga os
comentários de coluna junto com a estrutura anterior. Por isso a documentação dos
campos vem imediatamente após a criação, e não no notebook de catálogo: ela faz
parte da criação da tabela, não é um passo posterior.

In [0]:
%sql
ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN data
  COMMENT 'Data da medicao em violacao.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN hora
  COMMENT 'Hora do dia da medicao em violacao. Dominio: 0 a 23.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN id_subsistema
  COMMENT 'Codigo do subsistema onde a violacao ocorreu. Dominio: NE, N, S, SE.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN nom_subsistema
  COMMENT 'Nome do subsistema por extenso.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN geracao_mwmed
  COMMENT 'Soma das quatro fontes de geracao no instante, em MWmed.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN carga_mwmed
  COMMENT 'Carga verificada no instante, em MWmed.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN intercambio_mwmed
  COMMENT 'Intercambio liquido no instante, em MWmed. Positivo indica exportacao, negativo importacao.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN residuo_mwmed
  COMMENT 'Residuo da identidade do balanco: geracao menos carga menos intercambio. Esperado proximo de zero; valores altos indicam inconsistencia na origem.';

ALTER TABLE workspace.gold.qa_violacoes_balanco ALTER COLUMN data_verificacao
  COMMENT 'Momento em que a verificacao de qualidade foi executada. Metadado de auditoria.';

## 4. Contrato de dados

A verificação vira **regra permanente** do pipeline.

Uma checagem pontual informa o estado de hoje. Um contrato de dados impede que uma carga futura seja considerada aprovada para consumo sem investigação: se uma anomalia nova surgir — ao incluir 2026, por exemplo — a execução é interrompida com erro explícito, em vez de seguir adiante com dado inconsistente.

É a diferença entre **verificar uma vez** e **garantir sempre**.

O contrato verifica **quais** são as violações, e não apenas quantas. Uma contagem igual poderia esconder a troca de uma violação antiga por uma nova: se duas das 24 conhecidas desaparecessem e surgissem duas em outro subsistema, o total continuaria 24.

In [0]:
from pyspark.sql import functions as F

# Anomalia conhecida e documentada na seção 3: subsistema Sul, 14/09/2022, 24 horas.
SUBSISTEMA_CONHECIDO = "S"
DATA_CONHECIDA = "2022-09-14"
HORAS_CONHECIDAS = 24

viol = spark.table("workspace.gold.qa_violacoes_balanco")
e_conhecida = ((F.col("id_subsistema") == SUBSISTEMA_CONHECIDO)
               & (F.col("data") == F.to_date(F.lit(DATA_CONHECIDA))))

conhecidas = viol.filter(e_conhecida).count()
novas = viol.filter(~e_conhecida)
n_novas = novas.count()

print(f"Violações da anomalia conhecida (Sul, 14/09/2022): {conhecidas} de {HORAS_CONHECIDAS}")
print(f"Violações fora da anomalia conhecida             : {n_novas}")

if n_novas > 0:
    display(novas)
    raise ValueError(f"Balanço energético: {n_novas} violações novas, fora da anomalia documentada. Investigar antes de publicar.")

if conhecidas != HORAS_CONHECIDAS:
    raise ValueError(
        f"A anomalia documentada mudou: {conhecidas} violações em vez de {HORAS_CONHECIDAS}. "
        "A fonte pode ter revisado os dados; atualizar a documentação antes de publicar.")

print("\nContrato de dados OK: apenas a anomalia documentada está presente.")

Violações da anomalia conhecida (Sul, 14/09/2022): 24 de 24
Violações fora da anomalia conhecida             : 0

Contrato de dados OK: apenas a anomalia documentada está presente.


# PARTE 2 — Análise

Com a qualidade verificada, as cinco perguntas de negócio definidas na etapa de objetivo são respondidas por consultas SQL sobre o esquema estrela.

---

## Pergunta 1

**Como evoluiu a participação de cada fonte na matriz de geração, por subsistema, entre 2019 e 2025?**

A consulta agrega a geração por ano, subsistema e fonte, e calcula a participação percentual de cada fonte dentro do seu subsistema em cada ano. A unidade é TWh — o somatório de MWmed ao longo das horas resulta em MWh, dividido por um milhão.

In [0]:
%sql
WITH base AS (
  SELECT t.ano, s.nom_subsistema, f.nom_fonte,
         SUM(g.val_geracao_mwmed) AS mwh
  FROM workspace.gold.fato_geracao g
  JOIN workspace.gold.dim_tempo      t ON t.sk_tempo      = g.sk_tempo
  JOIN workspace.gold.dim_subsistema s ON s.sk_subsistema = g.sk_subsistema
  JOIN workspace.gold.dim_fonte      f ON f.sk_fonte      = g.sk_fonte
  GROUP BY 1, 2, 3
)
SELECT ano, nom_subsistema, nom_fonte,
       ROUND(mwh / 1e6, 2)                                                     AS energia_twh,
       ROUND(100.0 * mwh / SUM(mwh) OVER (PARTITION BY ano, nom_subsistema), 2) AS pct_matriz
FROM base
ORDER BY nom_subsistema, ano, nom_fonte;

ano,nom_subsistema,nom_fonte,energia_twh,pct_matriz
2019,NORDESTE,Eólica,46.11,52.41
2019,NORDESTE,Fotovoltaica,2.86,3.25
2019,NORDESTE,Hidráulica,21.55,24.49
2019,NORDESTE,Térmica,17.46,19.85
2020,NORDESTE,Eólica,46.34,46.46
2020,NORDESTE,Fotovoltaica,3.47,3.48
2020,NORDESTE,Hidráulica,37.52,37.61
2020,NORDESTE,Térmica,12.42,12.45
2021,NORDESTE,Eólica,64.2,51.76
2021,NORDESTE,Fotovoltaica,5.24,4.23


### Resposta

A matriz nacional se transformou em sete anos:

| Fonte | 2019 | 2025 | Variação |
|---|---|---|---|
| Hidráulica | 409,7 TWh (72,5%) | 403,8 TWh (57,7%) | −14,8 p.p. |
| Térmica | 97,6 TWh (17,3%) | 89,4 TWh (12,8%) | −4,5 p.p. |
| Eólica | 53,4 TWh (9,4%) | 115,4 TWh (16,5%) | +7,1 p.p. |
| Fotovoltaica | 4,4 TWh (0,8%) | 91,8 TWh (13,1%) | +12,3 p.p. |
| **Total** | **565,1 TWh** | **700,4 TWh** | **+23,9%** |

A geração fotovoltaica multiplicou-se por **21** no período e a eólica mais que dobrou. A hidráulica permaneceu estável em valor absoluto — toda a expansão da matriz veio de fontes renováveis não hidráulicas.

O caso do **Nordeste** é o mais acentuado. O subsistema já entrou no período com perfil atípico: em 2019 a eólica sozinha respondia por 52,4% de sua geração. Em 2025, eólica e solar somam 79,5%, a hidráulica caiu para 16,5% e a térmica recuou de 19,9% para 3,9%. Em sete anos o Nordeste deixou de ser um sistema hidrotérmico e passou a ser eólico-solar.

---

**Ressalva acrescentada após a extensão (notebook 07).** O fator de 21 vezes reflete fielmente o que o ONS publica, mas não corresponde a crescimento físico puro. A análise de capacidade instalada identificou uma quebra estrutural no Balanço de Energia em maio de 2023, compatível com alteração de escopo ou de metodologia, a partir da qual a geração fotovoltaica registrada passou a incluir uma parcela que antes não constava. Parte do salto é expansão real do parque e parte é ampliação do escopo contábil. O crescimento da eólica, verificado pelo mesmo método, não apresenta essa descontinuidade.

---

## Pergunta 2

**O crescimento da geração eólica e fotovoltaica reduziu a dependência de geração térmica?**

A consulta consolida a geração anual nacional por fonte, permitindo comparar trajetórias em valor absoluto — e não apenas em participação, que é onde a resposta se torna interessante.

In [0]:
%sql
WITH anual AS (
  SELECT t.ano, f.nom_fonte, SUM(g.val_geracao_mwmed) AS mwh
  FROM workspace.gold.fato_geracao g
  JOIN workspace.gold.dim_tempo t ON t.sk_tempo = g.sk_tempo
  JOIN workspace.gold.dim_fonte f ON f.sk_fonte = g.sk_fonte
  GROUP BY 1, 2
)
SELECT ano,
  ROUND(MAX(CASE WHEN nom_fonte = 'Hidráulica'   THEN mwh END) / 1e6, 1) AS hidraulica_twh,
  ROUND(MAX(CASE WHEN nom_fonte = 'Térmica'      THEN mwh END) / 1e6, 1) AS termica_twh,
  ROUND(MAX(CASE WHEN nom_fonte = 'Eólica'       THEN mwh END) / 1e6, 1) AS eolica_twh,
  ROUND(MAX(CASE WHEN nom_fonte = 'Fotovoltaica' THEN mwh END) / 1e6, 1) AS solar_twh,
  ROUND(SUM(mwh) / 1e6, 1)                                               AS total_twh
FROM anual
GROUP BY ano ORDER BY ano;

ano,hidraulica_twh,termica_twh,eolica_twh,solar_twh,total_twh
2019,409.7,97.6,53.4,4.4,565.1
2020,404.6,90.2,54.3,5.2,554.4
2021,372.7,141.9,72.3,7.4,594.4
2022,437.9,76.3,81.2,12.1,607.5
2023,442.1,72.3,95.4,43.0,652.8
2024,428.3,85.8,107.5,73.3,695.0
2025,403.8,89.4,115.4,91.8,700.4


### Resposta

A resposta depende de distinguir **valor absoluto** de **participação**, e essa distinção é o ponto central.

Em valor absoluto a térmica recuou pouco: de 97,6 TWh em 2019 para 89,4 TWh em 2025, queda de 8,4%. Em participação, caiu de 17,3% para 12,8% — porque a geração total cresceu 135,3 TWh no período.

A decomposição do crescimento fecha com precisão:

| Componente | TWh |
|---|---|
| Crescimento de eólica e fotovoltaica | +149,4 |
| Crescimento da demanda total | −135,3 |
| **Geração deslocada de outras fontes** | **14,1** |
| — deslocada da térmica | 8,2 |
| — deslocada da hidráulica | 5,9 |

As fontes intermitentes **absorveram integralmente o crescimento da demanda e ainda deslocaram 14,1 TWh** de geração preexistente.

O ano de **2021** demonstra o papel que a térmica ainda desempenha: a hidráulica caiu para 372,7 TWh, o menor valor da série, enquanto a térmica saltou para 141,9 TWh, 57% acima da média do período. É a crise hídrica de 2021 registrada no dado operativo — evidência de que a térmica, apesar da participação declinante, permanece como seguro do sistema contra falha hidrológica.

---

## Pergunta 3

**Qual subsistema apresenta hoje a maior dependência de fontes intermitentes?**

A consulta usa os indicadores `flag_intermitente` e `flag_renovavel` da dimensão de fonte para calcular, no ano mais recente da série, a participação de cada categoria na geração de cada subsistema.

Fontes **intermitentes** são aquelas que dependem de condições naturais no instante — eólica e fotovoltaica — e por isso não podem ser despachadas sob demanda.

In [0]:
%sql
WITH base AS (
  SELECT s.nom_subsistema, f.flag_intermitente, f.flag_renovavel,
         SUM(g.val_geracao_mwmed) AS mwh
  FROM workspace.gold.fato_geracao g
  JOIN workspace.gold.dim_tempo      t ON t.sk_tempo      = g.sk_tempo
  JOIN workspace.gold.dim_subsistema s ON s.sk_subsistema = g.sk_subsistema
  JOIN workspace.gold.dim_fonte      f ON f.sk_fonte      = g.sk_fonte
  WHERE t.ano = 2025
  GROUP BY 1, 2, 3
)
SELECT nom_subsistema,
  ROUND(100.0 * SUM(CASE WHEN flag_intermitente THEN mwh ELSE 0 END) / SUM(mwh), 2) AS pct_intermitente,
  ROUND(100.0 * SUM(CASE WHEN flag_renovavel    THEN mwh ELSE 0 END) / SUM(mwh), 2) AS pct_renovavel
FROM base
GROUP BY nom_subsistema
ORDER BY pct_intermitente DESC;

nom_subsistema,pct_intermitente,pct_renovavel
NORDESTE,79.52,96.05
SUL,17.61,90.62
SUDESTE/CENTRO-OESTE,13.8,83.78
NORTE,7.59,79.9


### Resposta

| Subsistema | Intermitentes | Renováveis |
|---|---|---|
| Nordeste | 79,5% | 96,1% |
| Sul | 17,6% | 90,6% |
| Sudeste/Centro-Oeste | 13,8% | 83,8% |
| Norte | 7,6% | 79,9% |

O **Nordeste** está em outro patamar: quase 80% de sua geração depende de recursos que variam no instante, sem possibilidade de despacho — mais de quatro vezes a participação do segundo colocado.

Isso o torna o subsistema com a matriz mais limpa do país e, simultaneamente, o mais exposto à variabilidade de curto prazo. É essa combinação que explica sua necessidade estrutural de forte interligação com os demais subsistemas, tema que reaparece na Pergunta 5.

---

## Pergunta 4

**Como carga e geração se comportam ao longo do dia? Existe descasamento entre o pico de demanda e o pico de geração solar?**

A consulta calcula o perfil médio horário nacional de 2025. A média é obtida somando todos os subsistemas em cada hora e dividindo pelo número de dias, o que resulta no valor nacional típico daquela hora do dia.

Esta é a pergunta que depende criticamente da fixação do fuso horário feita na camada Silver: um deslocamento de três horas inverteria a conclusão.

In [0]:
%sql
WITH ger AS (
  SELECT t.hora, f.nom_fonte,
         SUM(g.val_geracao_mwmed) / COUNT(DISTINCT t.data) AS media_nacional
  FROM workspace.gold.fato_geracao g
  JOIN workspace.gold.dim_tempo t ON t.sk_tempo = g.sk_tempo
  JOIN workspace.gold.dim_fonte f ON f.sk_fonte = g.sk_fonte
  WHERE t.ano = 2025
  GROUP BY 1, 2
),
car AS (
  SELECT t.hora, SUM(c.val_carga_mwmed) / COUNT(DISTINCT t.data) AS carga
  FROM workspace.gold.fato_carga c
  JOIN workspace.gold.dim_tempo t ON t.sk_tempo = c.sk_tempo
  WHERE t.ano = 2025
  GROUP BY 1
)
SELECT c.hora,
  ROUND(c.carga, 0)                                                              AS carga_mwmed,
  ROUND(MAX(CASE WHEN g.nom_fonte = 'Fotovoltaica' THEN g.media_nacional END), 0) AS solar,
  ROUND(MAX(CASE WHEN g.nom_fonte = 'Eólica'       THEN g.media_nacional END), 0) AS eolica,
  ROUND(MAX(CASE WHEN g.nom_fonte = 'Térmica'      THEN g.media_nacional END), 0) AS termica,
  ROUND(MAX(CASE WHEN g.nom_fonte = 'Hidráulica'   THEN g.media_nacional END), 0) AS hidraulica
FROM car c JOIN ger g ON g.hora = c.hora
GROUP BY c.hora, c.carga
ORDER BY c.hora;

hora,carga_mwmed,solar,eolica,termica,hidraulica
0,75668.0,4.0,17233.0,10375.0,48386.0
1,72376.0,4.0,17132.0,10278.0,45272.0
2,70167.0,4.0,16945.0,10253.0,43263.0
3,68896.0,4.0,16750.0,10246.0,42181.0
4,68658.0,22.0,16515.0,10242.0,42161.0
5,69642.0,586.0,16122.0,10226.0,42985.0
6,71906.0,5470.0,14471.0,10152.0,42096.0
7,75156.0,13939.0,12574.0,10007.0,38919.0
8,78221.0,20745.0,11102.0,9918.0,36756.0
9,79627.0,25901.0,9382.0,9854.0,34814.0


### Resposta

| Hora | Carga | Solar | Eólica | Térmica | Hidráulica |
|---|---|---|---|---|---|
| 04 | 68.658 | 22 | 16.515 | 10.242 | 42.161 |
| 11 | 82.160 | **31.430** | 7.253 | 9.824 | 34.010 |
| 12 | 80.598 | 31.118 | **6.737** | 9.819 | **33.306** |
| 19 | **89.975** | 21 | 15.532 | 10.521 | **64.278** |
| 23 | 80.016 | 5 | 17.183 | 10.500 | 52.722 |

**O pico de carga ocorre às 19h (89.975 MWmed); o pico solar às 11h (31.430 MWmed).** São **oito horas de defasagem**. No instante de maior demanda do sistema, a fotovoltaica entrega 21 MWmed — 0,02% da carga. No pico solar, ela cobre 38% de toda a carga nacional.

Três observações completam o quadro.

**A eólica compensa parcialmente.** Sua curva é anticorrelata à solar: 17.233 MWmed à meia-noite contra 6.737 ao meio-dia. O regime de ventos brasileiro é mais intenso à noite, o que suaviza o descasamento — característica favorável que não se verifica em todas as matrizes.

**A hidráulica é quem equilibra o sistema.** Cai para 33.306 MWmed no pico solar e sobe para 64.278 às 19h: uma rampa de aproximadamente **31.000 MWmed em sete horas**, quase dobrando a produção.

**A térmica opera como base, não como flexibilidade.** Varia apenas 7% ao longo das 24 horas, entre 9.819 e 10.552 MWmed.

A implicação operacional é que o Brasil usa seus reservatórios hidrelétricos para desempenhar a função que outros sistemas atribuem a baterias e usinas de partida rápida. É uma vantagem estrutural da matriz brasileira — e, ao mesmo tempo, uma dependência: a capacidade de integrar mais geração fotovoltaica está vinculada à disponibilidade hidrológica dos reservatórios.

---

## Pergunta 5

**Quais subsistemas são exportadores ou importadores líquidos de energia, e essa posição mudou no período?**

A consulta calcula o intercâmbio líquido médio de cada subsistema por ano. A interpretação do sinal — positivo para exportação, negativo para importação — foi determinada na Parte 1, a partir da identidade do balanço energético, já que o dicionário do ONS não a explicita.

In [0]:
%sql
SELECT t.ano, s.nom_subsistema,
  ROUND(AVG(c.val_intercambio_mwmed), 0) AS intercambio_medio_mwmed,
  CASE WHEN AVG(c.val_intercambio_mwmed) > 0 THEN 'Exportador' ELSE 'Importador' END AS posicao
FROM workspace.gold.fato_carga c
JOIN workspace.gold.dim_tempo      t ON t.sk_tempo      = c.sk_tempo
JOIN workspace.gold.dim_subsistema s ON s.sk_subsistema = c.sk_subsistema
GROUP BY t.ano, s.nom_subsistema
ORDER BY s.nom_subsistema, t.ano;

ano,nom_subsistema,intercambio_medio_mwmed,posicao
2019,NORDESTE,-561.0,Importador
2020,NORDESTE,1056.0,Exportador
2021,NORDESTE,2763.0,Exportador
2022,NORDESTE,3728.0,Exportador
2023,NORDESTE,4272.0,Exportador
2024,NORDESTE,5242.0,Exportador
2025,NORDESTE,6264.0,Exportador
2019,NORTE,3957.0,Exportador
2020,NORTE,4183.0,Exportador
2021,NORTE,5298.0,Exportador


### Resposta

| Subsistema | 2019 | 2021 | 2023 | 2025 | Trajetória |
|---|---|---|---|---|---|
| Nordeste | −561 | +2.763 | +4.272 | **+6.264** | Importador → exportador |
| Norte | +3.957 | +5.298 | +2.667 | +2.169 | Exportador em declínio |
| Sudeste/Centro-Oeste | −1.664 | −5.193 | −5.107 | −5.640 | Importador crescente |
| Sul | −1.807 | −3.542 | −1.022 | −2.446 | Importador oscilante |

A mudança é estrutural. O **Nordeste** era importador líquido em 2019 e tornou-se exportador já em 2020, alcançando +6.264 MWmed em 2025 — onze vezes o déficit que apresentava no início da série. É a consequência direta da expansão eólica e solar documentada na Pergunta 1: o subsistema passou a gerar muito além da própria carga e exporta o excedente.

Na outra ponta, o **Sudeste/Centro-Oeste**, que concentra a maior carga do país, aprofundou sua posição importadora de −1.664 para −5.640 MWmed. O sentido histórico do fluxo de energia no Brasil se inverteu: hoje o Nordeste abastece o Sudeste.

**Verificação adicional:** a soma dos quatro subsistemas em cada ano aproxima-se de zero (−75 MWmed em 2019, +347 em 2025), como esperado de um sistema fechado. O resíduo remanescente pode envolver perdas, convenções contábeis e arredondamentos; os dados disponíveis não permitem separá-los.

---

# Discussão geral

As cinco respostas convergem para um mesmo processo. A matriz elétrica brasileira não apenas incorporou fontes renováveis — ela **reorganizou sua geografia e sua dinâmica operacional**.

Na **dimensão geográfica**, a expansão eólica e solar concentrou-se no Nordeste, que deixou de ser deficitário e passou a abastecer o Sudeste. Isso inverteu o sentido histórico dos fluxos e transferiu para a rede de transmissão uma função que antes era menos crítica.

Na **dimensão temporal**, a entrada da solar criou um descasamento diário de oito horas entre oferta e demanda, absorvido quase integralmente pela hidráulica por meio de rampas de 31 GW. A térmica, apesar de perder participação, permanece como seguro contra falha hidrológica — o que 2021 demonstrou de forma inequívoca.

A conclusão para quem planeja o sistema é que o limite para integrar mais geração intermitente no Brasil **não está na capacidade de gerar**, e sim na capacidade de **transportar** essa energia do Nordeste para os centros de carga e de **armazenar** flexibilidade para as horas sem sol. Ambas as restrições aparecem nos dados analisados.

---

### Limitações

A análise cobre o recorte 2019–2025 e o dataset de Balanço de Energia nos Subsistemas, que agrega a geração por fonte sem detalhar usinas.

A ausência de dados de capacidade instalada, registrada como limitação na primeira versão, foi resolvida na extensão do notebook 07, que reconstruiu a série histórica de capacidade a partir das datas de entrada em operação e desativação. O fator de capacidade passou a ser calculável para hidráulica, eólica e térmica em todo o período.

Para a fotovoltaica, a extensão revelou uma quebra estrutural no Balanço a partir de maio de 2023, compatível com alteração de escopo ou metodologia. Isso torna o indicador comparável apenas até 2022 e impõe a ressalva registrada na Pergunta 1.